In [0]:

import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

# Audit configuration
PIPELINE_NAME = "retail_data_pipeline"
TASK_NAME = "04_gold_analytics"
AUDIT_TABLE = "workspace.retail_training.pipeline_audit_log"

RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(timezone.utc)

ROWS_PROCESSED = 0

print(f"Task audit ID: {RUN_ID}")

# 1. Read Silver sales
silver = spark.table(
    "workspace.retail_training.silver_sales"
)

# 2. Validate Silver data before building Gold
invalid_rows = silver.filter(
    F.col("sale_id").isNull()
    | F.col("sale_date").isNull()
    | F.col("quantity").isNull()
    | (F.col("quantity") <= 0)
    | F.col("unit_price").isNull()
    | (F.col("unit_price") < 0)
    | F.col("revenue").isNull()
).count()

if invalid_rows > 0:
    raise RuntimeError(
        f"Gold analytics stopped: {invalid_rows} invalid Silver rows found."
    )

print("Silver source validation passed.")


Task audit ID: 53db96f6-8d75-43fa-ab82-854f761decae
Silver source validation passed.


In [0]:

def write_audit_record(
    run_status,
    start_time,
    end_time,
    rows_processed=None,
    error_message=None
):
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_id", StringType(), True),
        StructField("run_status", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("logged_at", TimestampType(), True)
    ])

    audit_record = [(
        PIPELINE_NAME,
        TASK_NAME,
        RUN_ID,
        run_status,
        start_time,
        end_time,
        rows_processed,
        error_message[:4000] if error_message else None,
        end_time
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        AUDIT_TABLE
    )


In [0]:

try:
    # 1. Read Silver sales
    silver = spark.table(
        "workspace.retail_training.silver_sales"
    )

    # 2. Validate Silver data before building Gold
    invalid_rows = silver.filter(
        F.col("sale_id").isNull()
        | F.col("sale_date").isNull()
        | F.col("quantity").isNull()
        | (F.col("quantity") <= 0)
        | F.col("unit_price").isNull()
        | (F.col("unit_price") < 0)
        | F.col("revenue").isNull()
    ).count()

    if invalid_rows > 0:
        raise RuntimeError(
            f"Gold analytics stopped: {invalid_rows} invalid Silver rows found."
        )

    print("Silver source validation passed.")

    # 3. Aggregate Silver sales by month
    monthly_sales = (
        silver
        .withColumn(
            "sales_month",
            F.date_trunc("month", F.col("sale_date")).cast("date")
        )
        .groupBy("sales_month")
        .agg(
            F.countDistinct("sale_id").alias("total_sales"),
            F.sum("quantity").alias("total_quantity"),
            F.sum("revenue").alias("total_revenue")
        )
    )

    monthly_count = monthly_sales.count()

    # 4. Save the Gold monthly sales table
    (
        monthly_sales.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.retail_training.gold_monthly_sales")
    )

    print("Gold monthly sales table created successfully.")

    # 5. Aggregate Silver sales by product
    product_sales = (
        silver
        .groupBy("product_id")
        .agg(
            F.countDistinct("sale_id").alias("total_sales"),
            F.sum("quantity").alias("total_quantity"),
            F.sum("revenue").alias("total_revenue")
        )
    )

    product_count = product_sales.count()

    # 6. Save the Gold product sales table
    (
        product_sales.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.retail_training.gold_product_sales")
    )

    print("Gold product sales table created successfully.")

    # 7. Aggregate Silver sales by store
    store_sales = (
        silver
        .groupBy("store_id")
        .agg(
            F.countDistinct("sale_id").alias("total_sales"),
            F.sum("quantity").alias("total_quantity"),
            F.sum("revenue").alias("total_revenue")
        )
    )

    store_count = store_sales.count()

    # 8. Save the Gold store sales table
    (
        store_sales.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.retail_training.gold_store_sales")
    )

    print("Gold store sales table created successfully.")

    # Count the output rows across all three Gold tables
    ROWS_PROCESSED = (
        monthly_count + product_count + store_count
    )

    # 9. Display the summaries
    print("=== GOLD MONTHLY SALES ===")
    spark.sql("""
        SELECT *
        FROM workspace.retail_training.gold_monthly_sales
        ORDER BY sales_month
    """).show()

    print("=== GOLD PRODUCT SALES ===")
    spark.sql("""
        SELECT *
        FROM workspace.retail_training.gold_product_sales
        ORDER BY product_id
    """).show()

    print("=== GOLD STORE SALES ===")
    spark.sql("""
        SELECT *
        FROM workspace.retail_training.gold_store_sales
        ORDER BY store_id
    """).show()

    print("Gold analytics notebook completed.")

except Exception as e:
    end_time = datetime.now(timezone.utc)

    try:
        write_audit_record(
            run_status="FAILED",
            start_time=START_TIME,
            end_time=end_time,
            rows_processed=ROWS_PROCESSED,
            error_message=str(e)
        )
        print("FAILED audit record written.")
    except Exception as audit_error:
        print(f"WARNING: Could not write failure audit record: {audit_error}")

    raise

else:
    end_time = datetime.now(timezone.utc)

    write_audit_record(
        run_status="SUCCESS",
        start_time=START_TIME,
        end_time=end_time,
        rows_processed=ROWS_PROCESSED,
        error_message=None
    )

    print(
        f"SUCCESS audit record written: "
        f"{ROWS_PROCESSED} total Gold output rows."
    )


Silver source validation passed.
Gold monthly sales table created successfully.
Gold product sales table created successfully.
Gold store sales table created successfully.
=== GOLD MONTHLY SALES ===
+-----------+-----------+--------------+-------------+
|sales_month|total_sales|total_quantity|total_revenue|
+-----------+-----------+--------------+-------------+
| 2026-01-01|         38|           102|    986200.00|
| 2026-02-01|         33|            76|    552000.00|
| 2026-03-01|         34|            88|    800000.00|
| 2026-04-01|          5|            12|     33000.00|
+-----------+-----------+--------------+-------------+

=== GOLD PRODUCT SALES ===
+----------+-----------+--------------+-------------+
|product_id|total_sales|total_quantity|total_revenue|
+----------+-----------+--------------+-------------+
|         1|          9|            20|    815000.00|
|         2|         15|            41|     29200.00|
|         3|          9|            27|     40400.00|
|        